
# Búsqueda adversarial: algoritmo Minimax
---

## 1. Idea central: Tres en raya

En BFS, DFS o A* buscamos una ruta hacia un objetivo.  
En un juego ocurre algo diferente:

> después de que nosotros elegimos una acción, **otro jugador también elige**.

Por tanto, no basta con preguntarnos:

> ¿Cuál es la mejor jugada que puedo hacer?

También debemos considerar:

> ¿Qué hará mi oponente si intenta perjudicarme?

Minimax modela dos jugadores:

- **MAX:** intenta obtener el valor más alto posible.
- **MIN:** intenta obtener el valor más bajo posible.

Supondremos inicialmente que ambos jugadores actúan de manera racional.

## 2. Elementos de un problema adversarial

Un juego puede describirse mediante:

- **Estado:** configuración actual del juego.
- **Jugador actual:** indica quién debe mover.
- **Acciones:** movimientos legales desde el estado actual.
- **Resultado:** estado que se obtiene al aplicar una acción.
- **Estado terminal:** posición en la que la partida ha terminado.
- **Utilidad:** valor numérico asociado al resultado final.

Una convención sencilla puede ser:

| Resultado para MAX | Utilidad |
|---|---:|
| Victoria | `+1` |
| Empate | `0` |
| Derrota | `-1` |

En ejemplos más generales, la utilidad puede tomar cualquier valor numérico.

# 3. Primer árbol de juego

Consideremos el siguiente árbol.

- La raíz `A` pertenece a **MAX**.
- En el siguiente nivel juega **MIN**.
- Las hojas contienen valores de utilidad.

```text
                    A  (MAX)
              /         |         \
          B (MIN)    C (MIN)    D (MIN)
          / | \       / | \       / | \
         3  5  2     9  1  4     6  7  8
```

Pregunta:

> Si MIN juega racionalmente, ¿qué opción debería escoger MAX desde `A`?

In [1]:
arbol = {
    "A": ["B", "C", "D"],
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
    "D": ["D1", "D2", "D3"],
}

utilidades = {
    "B1": 3, "B2": 5, "B3": 2,
    "C1": 9, "C2": 1, "C3": 4,
    "D1": 6, "D2": 7, "D3": 8,
}

arbol, utilidades

({'A': ['B', 'C', 'D'],
  'B': ['B1', 'B2', 'B3'],
  'C': ['C1', 'C2', 'C3'],
  'D': ['D1', 'D2', 'D3']},
 {'B1': 3,
  'B2': 5,
  'B3': 2,
  'C1': 9,
  'C2': 1,
  'C3': 4,
  'D1': 6,
  'D2': 7,
  'D3': 8})

# 4. Razonamiento antes del algoritmo

Analicemos primero cada nodo de MIN.

### Nodo B

MIN puede elegir entre:

$$3,\ 5,\ 2$$

Por tanto:

$$\min(3,5,2)=2$$

### Nodo C

$$\min(9,1,4)=1$$

### Nodo D

$$\min(6,7,8)=6$$

La raíz pertenece a MAX, así que compara:

$$\max(2,1,6)=6$$

Por tanto, MAX debería elegir la rama `D`.

Esta idea es exactamente la que implementa **Minimax**.

# 5. Algoritmo Minimax

La definición recursiva puede escribirse como:

$$
V(s)=
\begin{cases}
U(s) & \text{si }s\text{ es terminal}\\
\max_{s' \in Sucesores(s)} V(s') & \text{si juega MAX}\\
\min_{s' \in Sucesores(s)} V(s') & \text{si juega MIN}
\end{cases}
$$

La recursión baja hasta los estados terminales y después los valores se
**propagan hacia arriba**.

In [2]:
def minimax(nodo, es_max, arbol, utilidades):
    # Caso base: nodo terminal
    if nodo in utilidades:
        return utilidades[nodo]

    valores = []

    for hijo in arbol[nodo]:
        valor = minimax(hijo, not es_max, arbol, utilidades)
        valores.append(valor)

    if es_max:
        return max(valores)
    else:
        return min(valores)


valor_raiz = minimax("A", True, arbol, utilidades)
valor_raiz

6

## 5.1 Obtener también la mejor jugada

Conocer el valor del estado es útil, pero normalmente necesitamos además saber:

> **¿qué acción debe ejecutar el jugador?**

La siguiente función devuelve el valor Minimax y el hijo seleccionado.

In [3]:
def mejor_jugada_minimax(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo], None

    opciones = []

    for hijo in arbol[nodo]:
        valor = minimax(hijo, not es_max, arbol, utilidades)
        opciones.append((valor, hijo))

    if es_max:
        valor, hijo = max(opciones, key=lambda x: x[0])
    else:
        valor, hijo = min(opciones, key=lambda x: x[0])

    return valor, hijo


valor, jugada = mejor_jugada_minimax("A", True, arbol, utilidades)

print("Valor Minimax:", valor)
print("Mejor jugada para MAX:", jugada)

Valor Minimax: 6
Mejor jugada para MAX: D


# 6. Minimax paso a paso

Para comprender mejor el algoritmo observaremos la recursión.

La sangría permite identificar la profundidad en el árbol.

In [4]:
def minimax_debug(nodo, es_max, arbol, utilidades, profundidad=0):
    sangria = "    " * profundidad
    jugador = "MAX" if es_max else "MIN"

    if nodo in utilidades:
        print(f"{sangria}{nodo}: terminal -> utilidad {utilidades[nodo]}")
        return utilidades[nodo]

    print(f"{sangria}{nodo}: turno de {jugador}")
    valores = []

    for hijo in arbol[nodo]:
        valor = minimax_debug(
            hijo,
            not es_max,
            arbol,
            utilidades,
            profundidad + 1
        )
        valores.append(valor)

    if es_max:
        resultado = max(valores)
    else:
        resultado = min(valores)

    print(f"{sangria}{nodo}: {jugador} selecciona {resultado}")
    return resultado


minimax_debug("A", True, arbol, utilidades)

A: turno de MAX
    B: turno de MIN
        B1: terminal -> utilidad 3
        B2: terminal -> utilidad 5
        B3: terminal -> utilidad 2
    B: MIN selecciona 2
    C: turno de MIN
        C1: terminal -> utilidad 9
        C2: terminal -> utilidad 1
        C3: terminal -> utilidad 4
    C: MIN selecciona 1
    D: turno de MIN
        D1: terminal -> utilidad 6
        D2: terminal -> utilidad 7
        D3: terminal -> utilidad 8
    D: MIN selecciona 6
A: MAX selecciona 6


6

### Preguntas de análisis

1. ¿Por qué MAX no selecciona directamente la hoja con valor `9`?

Porque esa hoja está bajo el nodo C, y si X va por ahí, es MIN quien decide entre 9, 1 y 4 — y obviamente MIN va a elegir 1, no 9. O sea, el 9 nunca llega a pasar, es una ilusión.

2. ¿Qué supone Minimax sobre el comportamiento del adversario?

Que juega perfecto y siempre en tu contra. No asume que se equivoca ni que es despistado, asume el peor escenario posible.

3. ¿Qué ocurriría si MIN no escogiera siempre la opción de menor valor?

Entonces el valor que calculamos deja de ser confiable, porque estaríamos siendo pesimistas de más. Si el rival juega peor de lo esperado, en la práctica te puede ir mejor que lo que predijo el algoritmo.

4. ¿Por qué los valores se calculan desde las hojas hacia la raíz?

Porque para saber qué tan buena es una jugada en la raíz necesitas saber cómo termina el juego, y eso solo lo sabes al final. Así que primero resuelves lo de abajo y vas "subiendo" esa información.

5. ¿El valor de una hoja representa necesariamente una puntuación real del juego?

No necesariamente. En el ejemplo son números inventados para practicar, pero en tres en raya sí tienen un significado real (ganar, perder o empatar). En juegos más complejos, muchas veces son solo una estimación (heurística), no el resultado real.

# 7. Un árbol con mayor profundidad

Ahora utilizaremos un árbol de tres decisiones.

```text
MAX → MIN → MAX → utilidad
```

Esto permite observar que los roles se alternan en cada nivel.

In [5]:
arbol_profundo = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G1", "G2"],
}

utilidades_profundo = {
    "D1": 3, "D2": 5,
    "E1": 6, "E2": 9,
    "F1": 1, "F2": 2,
    "G1": 0, "G2": -1,
}

minimax_debug("A", True, arbol_profundo, utilidades_profundo)

A: turno de MAX
    B: turno de MIN
        D: turno de MAX
            D1: terminal -> utilidad 3
            D2: terminal -> utilidad 5
        D: MAX selecciona 5
        E: turno de MAX
            E1: terminal -> utilidad 6
            E2: terminal -> utilidad 9
        E: MAX selecciona 9
    B: MIN selecciona 5
    C: turno de MIN
        F: turno de MAX
            F1: terminal -> utilidad 1
            F2: terminal -> utilidad 2
        F: MAX selecciona 2
        G: turno de MAX
            G1: terminal -> utilidad 0
            G2: terminal -> utilidad -1
        G: MAX selecciona 0
    C: MIN selecciona 0
A: MAX selecciona 5


5

## 7.1 Contar nodos evaluados

En árboles pequeños Minimax resulta sencillo.  
Sin embargo, el número de posiciones posibles puede crecer rápidamente.

Contaremos cuántos nodos visita el algoritmo.

In [6]:
def minimax_contando(nodo, es_max, arbol, utilidades, contador):
    contador["visitados"] += 1

    if nodo in utilidades:
        contador["terminales"] += 1
        return utilidades[nodo]

    valores = [
        minimax_contando(hijo, not es_max, arbol, utilidades, contador)
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


contador = {"visitados": 0, "terminales": 0}
valor = minimax_contando(
    "A",
    True,
    arbol_profundo,
    utilidades_profundo,
    contador
)

print("Valor Minimax:", valor)
print("Nodos visitados:", contador["visitados"])
print("Hojas evaluadas:", contador["terminales"])

Valor Minimax: 5
Nodos visitados: 15
Hojas evaluadas: 8


# 8. Caso aplicado: juego de las piedras

Trabajaremos con un juego muy sencillo:

- Existe una pila con cierta cantidad de piedras.
- En cada turno un jugador puede retirar `1`, `2` o `3` piedras.
- El jugador que retira la **última piedra gana**.

Representaremos un estado como:

```python
(piedras_restantes, jugador)
```

donde:

- `jugador = 1` representa a MAX;
- `jugador = -1` representa a MIN.

In [7]:
MOVIMIENTOS = (1, 2, 3)

def movimientos_validos(piedras):
    return [m for m in MOVIMIENTOS if m <= piedras]


for n in range(1, 8):
    print(n, "piedras ->", movimientos_validos(n))

1 piedras -> [1]
2 piedras -> [1, 2]
3 piedras -> [1, 2, 3]
4 piedras -> [1, 2, 3]
5 piedras -> [1, 2, 3]
6 piedras -> [1, 2, 3]
7 piedras -> [1, 2, 3]


## 8.1 Utilidad del estado terminal

Cuando no quedan piedras, significa que el jugador anterior tomó la última.

Si el jugador que debe mover ahora es MAX, entonces MIN realizó la jugada anterior
y ganó. Por tanto, la utilidad para MAX es `-1`.

Si debe mover MIN, MAX realizó la jugada anterior y ganó. La utilidad es `+1`.

In [8]:
def minimax_piedras(piedras, turno_max):
    if piedras == 0:
        return -1 if turno_max else 1

    valores = []

    for retirar in movimientos_validos(piedras):
        valor = minimax_piedras(
            piedras - retirar,
            not turno_max
        )
        valores.append(valor)

    return max(valores) if turno_max else min(valores)


for piedras in range(1, 11):
    print(
        f"{piedras:2d} piedras -> valor Minimax:",
        minimax_piedras(piedras, True)
    )

 1 piedras -> valor Minimax: 1
 2 piedras -> valor Minimax: 1
 3 piedras -> valor Minimax: 1
 4 piedras -> valor Minimax: -1
 5 piedras -> valor Minimax: 1
 6 piedras -> valor Minimax: 1
 7 piedras -> valor Minimax: 1
 8 piedras -> valor Minimax: -1
 9 piedras -> valor Minimax: 1
10 piedras -> valor Minimax: 1


## 8.2 Encontrar la mejor jugada

Ahora determinaremos cuántas piedras debería retirar MAX.

In [9]:
def mejor_movimiento_piedras(piedras):
    opciones = []

    for retirar in movimientos_validos(piedras):
        valor = minimax_piedras(piedras - retirar, False)
        opciones.append((valor, retirar))

    mejor_valor, mejor_movimiento = max(opciones, key=lambda x: x[0])

    return {
        "retirar": mejor_movimiento,
        "valor": mejor_valor,
        "opciones": opciones,
    }


for piedras in range(1, 11):
    print(
        f"{piedras:2d} piedras ->",
        mejor_movimiento_piedras(piedras)
    )

 1 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1)]}
 2 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2)]}
 3 piedras -> {'retirar': 3, 'valor': 1, 'opciones': [(-1, 1), (-1, 2), (1, 3)]}
 4 piedras -> {'retirar': 1, 'valor': -1, 'opciones': [(-1, 1), (-1, 2), (-1, 3)]}
 5 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1), (-1, 2), (-1, 3)]}
 6 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2), (-1, 3)]}
 7 piedras -> {'retirar': 3, 'valor': 1, 'opciones': [(-1, 1), (-1, 2), (1, 3)]}
 8 piedras -> {'retirar': 1, 'valor': -1, 'opciones': [(-1, 1), (-1, 2), (-1, 3)]}
 9 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1), (-1, 2), (-1, 3)]}
10 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2), (-1, 3)]}


### Preguntas de análisis

1. ¿Qué cantidades iniciales de piedras representan una posición desfavorable para MAX?

Los múltiplos de 4: 4, 8, 12... En esos casos MAX pierde, mueva lo que mueva.

2. ¿Existe algún patrón?

Sí, se repite cada 4 piedras. Básicamente porque puedes quitar 1, 2 o 3, entonces el rival siempre puede "completar" a 4 la diferencia y dejarte de nuevo en un múltiplo de 4.

3. ¿Por qué algunas posiciones tienen valor `-1` incluso si MAX todavía dispone de varios movimientos?

Porque tener opciones no significa tener buenas opciones. Si estás en un múltiplo de 4, hagas lo que hagas el rival te devuelve a otro múltiplo de 4, y así hasta que te quedas sin piedras en su turno.

4. ¿Puede haber más de una jugada igualmente buena?

Sí, si dos movimientos distintos te dan el mismo valor minimax (por ejemplo ambos te llevan a ganar), da igual cuál elijas.

5. ¿Qué cambiaría si fuera obligatorio retirar únicamente `1` o `2` piedras?

El patrón cambia de "cada 4" a "cada 3", porque ahora el rival puede compensar como máximo 2, así que el múltiplo relevante es 3 (1+2). Las posiciones perdedoras pasarían a ser 3, 6, 9, 12...

## 9 Minimax con profundidad limitada

La siguiente versión admite:

- una profundidad máxima;
- una función de evaluación para estados no terminales.

Este esquema es mucho más cercano al utilizado en juegos reales.

In [10]:
def minimax_limitado(
    estado,
    profundidad,
    es_max,
    es_terminal,
    utilidad,
    sucesores,
    evaluar
):
    if es_terminal(estado):
        return utilidad(estado)

    if profundidad == 0:
        return evaluar(estado)

    valores = [
        minimax_limitado(
            hijo,
            profundidad - 1,
            not es_max,
            es_terminal,
            utilidad,
            sucesores,
            evaluar
        )
        for hijo in sucesores(estado)
    ]

    return max(valores) if es_max else min(valores)

# 10. Taller

Implemente Minimax para **Tres en raya (Tic-Tac-Toe)**.

Puede representar el tablero como una tupla de nueve posiciones:

```python
(
    "X", "O", " ",
    " ", "X", " ",
    "O", " ", " "
)
```

Suponga:

- `X` es MAX;
- `O` es MIN;
- victoria de `X`: `+1`;
- empate: `0`;
- victoria de `O`: `-1`.

Una convención sencilla puede ser:

| Resultado para MAX | Utilidad |
|---|---:|
| Victoria de `X`| `+1` |
| Empate | `0` |
| Victoria para  | `-1` |

Implemente como mínimo: 

```python
acciones(tablero)
resultado(tablero, accion, jugador)
terminal(tablero)
utilidad(tablero)
minimax_tictactoe(tablero, es_max)
```

In [11]:
# Implemente aquí Minimax para Tres en raya.

LINEAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),  # filas
    (0, 3, 6), (1, 4, 7), (2, 5, 8),  # columnas
    (0, 4, 8), (2, 4, 6),             # diagonales
]


def acciones(tablero):
    """Devuelve las posiciones (0-8) que todavía están vacías."""
    return [i for i, casilla in enumerate(tablero) if casilla == " "]


def resultado(tablero, accion, jugador):
    """Tablero resultante de colocar `jugador` ('X' u 'O') en `accion`."""
    nuevo_tablero = list(tablero)
    nuevo_tablero[accion] = jugador
    return tuple(nuevo_tablero)


def ganador(tablero):
    """Devuelve 'X', 'O' o None según si hay tres en línea."""
    for a, b, c in LINEAS:
        if tablero[a] != " " and tablero[a] == tablero[b] == tablero[c]:
            return tablero[a]
    return None


def terminal(tablero):
    """True si alguien ganó o si ya no quedan casillas vacías."""
    return ganador(tablero) is not None or len(acciones(tablero)) == 0


def utilidad(tablero):
    """+1 si gana X, -1 si gana O, 0 en caso de empate."""
    quien_gano = ganador(tablero)
    if quien_gano == "X":
        return 1
    elif quien_gano == "O":
        return -1
    else:
        return 0


def minimax_tictactoe(tablero, es_max):
    """Valor minimax del tablero. es_max=True indica que juega X (MAX)."""
    if terminal(tablero):
        return utilidad(tablero)

    jugador = "X" if es_max else "O"
    valores = [
        minimax_tictactoe(resultado(tablero, accion, jugador), not es_max)
        for accion in acciones(tablero)
    ]

    return max(valores) if es_max else min(valores)


def mejor_jugada_tictactoe(tablero, es_max):
    """Devuelve (valor_minimax, mejor_accion) para quien tiene el turno."""
    jugador = "X" if es_max else "O"
    opciones = [
        (minimax_tictactoe(resultado(tablero, accion, jugador), not es_max), accion)
        for accion in acciones(tablero)
    ]

    if es_max:
        return max(opciones, key=lambda x: x[0])
    else:
        return min(opciones, key=lambda x: x[0])


def imprimir_tablero(tablero):
    for i in range(0, 9, 3):
        print(" | ".join(tablero[i:i + 3]))
        if i < 6:
            print("-" * 9)

In [16]:
# Prueba rápida de las funciones básicas

tablero_vacio = (" ",) * 9
print("Tablero vacío:")
imprimir_tablero(tablero_vacio)
print("acciones:", acciones(tablero_vacio))
print("terminal:", terminal(tablero_vacio))

tablero_prueba = (
    "X", "O", "X",
    "X", "O", " ",
    " ", " ", "O",
)
print("\nTablero de prueba:")
imprimir_tablero(tablero_prueba)
print("acciones:", acciones(tablero_prueba))
print("¿ganador?:", ganador(tablero_prueba))
print("terminal:", terminal(tablero_prueba))
print("utilidad:", utilidad(tablero_prueba))

Tablero vacío:
  |   |  
---------
  |   |  
---------
  |   |  
acciones: [0, 1, 2, 3, 4, 5, 6, 7, 8]
terminal: False

Tablero de prueba:
X | O | X
---------
X | O |  
---------
  |   | O
acciones: [5, 6, 7]
¿ganador?: None
terminal: False
utilidad: 0


### Para el juego de 3 en raya:

Construya un árbol de al menos tres niveles y:

1. asigne valores de utilidad a las hojas;
2. calcule manualmente los valores Minimax;
3. compruebe el resultado con Python;
4. indique la jugada elegida por MAX.

Complete la implementación propuesta y pruebe diferentes tableros.

### Para el juego de las piedras
Modifique las reglas para permitir retirar únicamente `1`, `2` o `4` piedras.

Analice:

- posiciones ganadoras;
- posiciones perdedoras;
- mejor movimiento para MAX.


### Pregunta final

**¿Por qué una decisión que parece buena de manera inmediata puede ser mala después de considerar la respuesta del adversario?**

### Uso de IA generativa

Si utiliza IA generativa, indique:

- herramienta utilizada;
- propósito de uso;
- partes de la solución en las que fue empleada.

### 10.1 Árbol de tres niveles (Tres en raya)

Partimos del siguiente tablero, en el que le toca mover a `X` (MAX) y quedan
exactamente tres casillas vacías (`6`, `7`, `8`), por lo que el árbol de juego
tiene **tres niveles** (X → O → X) hasta llegar a los estados terminales:

```text
X | O | X
---------
O | O | X
---------
  |   |
```

**Cálculo manual:**

- Si `X` juega en `8`: la columna `2-5-8` queda `X X X` → **victoria de X**,
  utilidad `+1` (el árbol termina antes, en el primer nivel).
- Si `X` juega en `6`:
  - `O` juega en `7` → columna `1-4-7` queda `O O O` → victoria de `O`,
    utilidad `-1`.
  - `O` juega en `8` → no hay ganador; a `X` solo le queda `7`, tablero
    lleno sin líneas completas → **empate**, utilidad `0`.
  - MIN elige el valor más bajo: `min(-1, 0) = -1`.
- Si `X` juega en `7`:
  - `O` juega en `6` → a `X` solo le queda `8`, columna `2-5-8` queda
    `X X X` → victoria de `X`, utilidad `+1`.
  - `O` juega en `8` → a `X` solo le queda `6`, tablero lleno sin líneas
    completas → empate, utilidad `0`.
  - MIN elige el valor más bajo: `min(1, 0) = 0`.

En la raíz, `X` (MAX) compara sus tres opciones:

$$\max(\,\underbrace{1}_{6\to8},\ \underbrace{-1}_{6},\ \underbrace{0}_{7}\,) = 1$$

Por tanto, la jugada óptima para `X` es colocar ficha en la casilla **`8`**,
que gana la partida de inmediato.

A continuación se verifica el resultado con Python, incluyendo una versión
de depuración (análoga a `minimax_debug`) para observar la recursión.

In [13]:
def minimax_tictactoe_debug(tablero, es_max, profundidad=0):
    sangria = "    " * profundidad
    jugador_turno = "MAX (X)" if es_max else "MIN (O)"

    if terminal(tablero):
        print(f"{sangria}terminal -> utilidad {utilidad(tablero)}")
        return utilidad(tablero)

    print(f"{sangria}turno de {jugador_turno}, acciones={acciones(tablero)}")
    jugador = "X" if es_max else "O"
    valores = []

    for accion in acciones(tablero):
        valor = minimax_tictactoe_debug(
            resultado(tablero, accion, jugador),
            not es_max,
            profundidad + 1,
        )
        valores.append(valor)

    resultado_final = max(valores) if es_max else min(valores)
    print(f"{sangria}{jugador_turno} selecciona valor {resultado_final}")
    return resultado_final


tablero_3_niveles = (
    "X", "O", "X",
    "O", "O", "X",
    " ", " ", " ",
)

imprimir_tablero(tablero_3_niveles)
print()
minimax_tictactoe_debug(tablero_3_niveles, True)

print()
valor, jugada = mejor_jugada_tictactoe(tablero_3_niveles, True)
print("Valor Minimax (coincide con el cálculo manual):", valor)
print("Mejor jugada para MAX (X):", jugada)

X | O | X
---------
O | O | X
---------
  |   |  

turno de MAX (X), acciones=[6, 7, 8]
    turno de MIN (O), acciones=[7, 8]
        terminal -> utilidad -1
        turno de MAX (X), acciones=[7]
            terminal -> utilidad 0
        MAX (X) selecciona valor 0
    MIN (O) selecciona valor -1
    turno de MIN (O), acciones=[6, 8]
        turno de MAX (X), acciones=[8]
            terminal -> utilidad 1
        MAX (X) selecciona valor 1
        turno de MAX (X), acciones=[6]
            terminal -> utilidad 0
        MAX (X) selecciona valor 0
    MIN (O) selecciona valor 0
    terminal -> utilidad 1
MAX (X) selecciona valor 1

Valor Minimax (coincide con el cálculo manual): 1
Mejor jugada para MAX (X): 8


### 10.2 Prueba con otros tableros

Probemos `minimax_tictactoe` con un tablero vacío (juego completo) y con un
tablero donde `O` está a punto de ganar, para confirmar que el algoritmo
detecta correctamente la mejor defensa/ataque en cada caso.

In [14]:
# Caso 1: tablero donde X debe bloquear a O para no perder
tablero_bloqueo = (
    "O", "O", " ",
    " ", "X", " ",
    " ", " ", " ",
)
imprimir_tablero(tablero_bloqueo)
valor, jugada = mejor_jugada_tictactoe(tablero_bloqueo, True)
print("Valor:", valor, "- Mejor jugada para X:", jugada, "(debe ser 2, para bloquear la fila)")

print()

# Caso 2: tablero vacío -> con juego perfecto de ambos lados el resultado es empate
tablero_vacio = (" ",) * 9
valor, jugada = mejor_jugada_tictactoe(tablero_vacio, True)
print("Valor Minimax desde el tablero vacío:", valor, "(0 = empate con juego óptimo)")
print("Una primera jugada óptima para X:", jugada)

O | O |  
---------
  | X |  
---------
  |   |  
Valor: 0 - Mejor jugada para X: 2 (debe ser 2, para bloquear la fila)

Valor Minimax desde el tablero vacío: 0 (0 = empate con juego óptimo)
Una primera jugada óptima para X: 0


### 10.3 Juego de las piedras: movimientos `1`, `2` o `4`

Modificamos las reglas para permitir retirar `1`, `2` o `4` piedras (ya **no**
se permite retirar `3`). Reutilizamos la misma lógica de `minimax_piedras`,
cambiando únicamente el conjunto de movimientos válidos.

In [15]:
MOVIMIENTOS_V2 = (1, 2, 4)


def movimientos_validos_v2(piedras):
    return [m for m in MOVIMIENTOS_V2 if m <= piedras]


def minimax_piedras_v2(piedras, turno_max):
    if piedras == 0:
        return -1 if turno_max else 1

    valores = [
        minimax_piedras_v2(piedras - retirar, not turno_max)
        for retirar in movimientos_validos_v2(piedras)
    ]

    return max(valores) if turno_max else min(valores)


def mejor_movimiento_piedras_v2(piedras):
    opciones = [
        (minimax_piedras_v2(piedras - retirar, False), retirar)
        for retirar in movimientos_validos_v2(piedras)
    ]
    mejor_valor, mejor_movimiento = max(opciones, key=lambda x: x[0])
    return {"retirar": mejor_movimiento, "valor": mejor_valor, "opciones": opciones}


print(f"{'piedras':>7} | {'valor':>5} | {'posición':>10} | mejor retiro")
for piedras in range(1, 13):
    valor = minimax_piedras_v2(piedras, True)
    posicion = "ganadora" if valor == 1 else "perdedora" if valor == -1 else "empate"
    info = mejor_movimiento_piedras_v2(piedras)
    print(f"{piedras:>7} | {valor:>+5} | {posicion:>10} | retirar {info['retirar']}")

piedras | valor |   posición | mejor retiro
      1 |    +1 |   ganadora | retirar 1
      2 |    +1 |   ganadora | retirar 2
      3 |    -1 |  perdedora | retirar 1
      4 |    +1 |   ganadora | retirar 1
      5 |    +1 |   ganadora | retirar 2
      6 |    -1 |  perdedora | retirar 1
      7 |    +1 |   ganadora | retirar 1
      8 |    +1 |   ganadora | retirar 2
      9 |    -1 |  perdedora | retirar 1
     10 |    +1 |   ganadora | retirar 1
     11 |    +1 |   ganadora | retirar 2
     12 |    -1 |  perdedora | retirar 1


**Análisis:**

- **Posiciones perdedoras para quien mueve** (valor `-1`): `3, 6, 9, 12, ...`
  — es decir, todos los **múltiplos de 3**. Con movimientos `{1, 2, 4}`,
  siempre es posible llevar al oponente a un múltiplo de 3 desde cualquier
  otro número (restando `1` o `2`), y desde un múltiplo de 3 cualquier
  movimiento (`1`, `2` o `4`) deja al rival en una posición no múltiplo de 3,
  desde la cual puede volver a forzarte a otro múltiplo de 3.
- **Posiciones ganadoras**: todas las que **no** son múltiplos de 3.
- **Mejor movimiento para MAX**: llevar siempre el número de piedras
  restantes a un múltiplo de 3 (retirando `1` si sobra `1` respecto al
  múltiplo, o `2` si sobran `2`).

### Pregunta final

**¿Por qué una decisión que parece buena de manera inmediata puede ser mala
después de considerar la respuesta del adversario?**

Porque una jugada solo se ve buena si la miramos aislada, sin pensar en lo que viene después. Pero el resultado del juego depende de toda la secuencia de jugadas, no solo de la tuya. Si el rival juega bien, va a responder de la forma que más te perjudique. Minimax justo hace eso: no evalúa "qué tan buena se ve mi jugada ahora", sino "qué tan buena queda después de que el rival responda lo mejor que puede". Por eso algo que parece un buen movimiento puede terminar siendo malo: abre una puerta que el rival va a aprovechar.

### Uso de IA generativa

- **Herramienta utilizada:** Claude.
- **Propósito de uso:** apoyo para entender mejor cómo aplicar la lógica de Minimax al caso de Tres en raya (representación del tablero, detección de líneas ganadoras, cómo alternar turnos en la recursión) y para revisar/verificar el árbol de ejemplo y la variante del juego de las piedras.
- **Partes de la solución en las que fue empleada:** sección 10. Taller — principalmente como guía y verificación, ajustando y completando el código a partir de las ideas discutidas.